# CUDA prompt-embedding experiments

Validate one reproducible single-objective experiment with complete survivor and rejected-attempt archives. The campaign controller runs a configured seed queue with isolated runners and owns final notebook capture, optional Drive upload and verified cleanup. See [the execution guide](../docs/azure.md).

This template defaults to execution disabled. Run from an installed working checkout, with the documented inference environment and packaged or external bounds. No cells install packages or start Azure compute.

In [ ]:
import os, sys
from pathlib import Path

# For Azure, the kernel specification also sets this before process startup.
if "torch" in sys.modules and not globals().get("EXPERIMENT_BOOTSTRAPPED", False):
    raise RuntimeError("Restart with NVIDIA_TF32_OVERRIDE=0 before importing PyTorch")
os.environ["NVIDIA_TF32_OVERRIDE"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
EXPERIMENT_BOOTSTRAPPED = True
checkout_hint = Path(os.environ.get("EVOLUTIONARY_CHECKOUT", Path.cwd())).resolve()
checkout = next((path for path in (checkout_hint, *checkout_hint.parents)
                 if (path / "setup.py").is_file()
                 and (path / "evolutionary_extensions/experiments/prompt_embedding/__init__.py").is_file()), None)
if checkout is None:
    raise RuntimeError("Set EVOLUTIONARY_CHECKOUT to your installed repository checkout")
sys.path.insert(0, str(checkout))

## Configuration
The tagged cell is the controller’s only parameter injection point. Paths and seeds are explicit; input bounds are validated before expensive model loading.

In [ ]:
CONFIG_PATH = checkout / "configs/examples/osga.json"
EXPERIMENT_PARAMETERS = None  # The controller injects one resolved experiment configuration.
RUN_EXPERIMENT = False

In [ ]:
import json
from evolutionary_extensions.experiments.prompt_embedding import ExperimentConfig, Runtime, load_bounds, run_experiment
if EXPERIMENT_PARAMETERS is None:
    document = json.loads(Path(CONFIG_PATH).read_text())
    parameters = {**document.get("deployment", {}), **document["experiment"]}
else:
    parameters = dict(EXPERIMENT_PARAMETERS)
for name in ("output_root", "bounds_file", "cache_dir", "initial_embeddings_file"):
    if parameters.get(name) is None:
        continue
    path = Path(parameters[name]).expanduser()
    parameters[name] = str(path if path.is_absolute() else checkout / path)
config = ExperimentConfig(**parameters)
bounds, bounds_manifest = load_bounds(config.bounds_file, config.bounds_source)
print("Configuration:", config)
print("Bounds:", bounds_manifest)
print(f"At most {config.num_generations} populations including initialization; at most {config.max_evaluations} fresh evaluations. OSGA may stop earlier.")

## Explicit experiment execution
One reusable CUDA pipeline/scorer serves successive runs within each isolated runner. Seed variation only after models load. CPU reference scoring uses an isolated process and identical images.

In [ ]:
PREFLIGHT_DIRECTORY = globals().get("PREFLIGHT_DIRECTORY", str(Path(config.output_root).parent / "preflight"))
if RUN_EXPERIMENT:
    if config.seed is None:
        import secrets
        from dataclasses import replace, asdict
        config = replace(config, seed=secrets.randbits(32))
        from evolutionary_extensions.persistence.packaging import atomic_json
        atomic_json(Path(config.output_root) / "standalone_manifest.json", {"seed": config.seed, "configuration": asdict(config)})
    config.guard()
    if "runtime" not in globals() or not runtime.compatible(config):
        runtime = Runtime(config)
    if (runtime.parity is None or runtime.parity.get("candidate_batch_size") != config.candidate_batch_size
        or runtime.parity.get("canonical_scoring", False) != config.canonical_scoring
        or runtime.parity.get("render_batch_size") != config.render_batch_size):
        runtime.check_parity(config, PREFLIGHT_DIRECTORY)
    if config.initialization == "local":
        runtime.check_anchor(config, Path(config.initial_embeddings_file).parent / "local-anchor-verification")


In [ ]:
run_result = None
if RUN_EXPERIMENT:
    config.guard()
    run_result = run_experiment(config, runtime)
    print(json.dumps({key: run_result[key] for key in ("run_id", "status", "completed_generations", "evaluation_count", "termination_reason", "best_fitness", "loop_seconds")}, indent=2))
else:
    print("Execution disabled. Set RUN_EXPERIMENT=True only for your finite experiment.")

## Results and artifact inspection
All attempt tensors/images are retained, including rejected and uncommitted offspring. Survivor snapshots keep evaluation-ID links. Figures summarize fitness, pressure, operators, timings, and token/pooled tensors; full projection analysis is performed offline.

In [ ]:
from IPython.display import display, Image
if run_result:
    run_dir = Path(run_result["run_dir"])
    display(Image(filename=str(run_dir / "plots/fitness.png")))
    display(Image(filename=str(run_dir / "media" / f"generation-{run_result['completed_generations'] - 1:03d}.png")))
    print("Survivor archive:", run_dir)
    print("All evaluations:", run_dir / "attempts")
    print("Offline analysis: notebooks/embedding_analysis.ipynb; select either or both archives.")

## Final notebook capture and persistence
The controller waits for kernel idle, saves this executed notebook, validates the run, and writes the artifact-completion marker. It then optionally packages/uploads the immutable folder and verifies a streamed Drive download before cleanup. Running this cell does not archive a notebook that is still executing. For manual execution, save the completed notebook and use the documented finalization command. Checkpoints export state; optimization resume is not implemented.

In [ ]:
if run_result:
    print("Inference/export status:", run_result["status"])
    print("Ready for controller finalization:", run_result["run_dir"])
    print("Termination limits and actual evaluation counts are retained, including unfinished attempts.")